# Simulações OpenDSS — IEEE 37 Barras
**Geração Fotovoltaica (PV), Armazenamento de Energia (BESS) e Operação Ilhada (Microrrede)**

---
### Escopo do Projeto:
* **Parte I (Impactos de GD):** Instalação de PV em ≥ 20% das barras, dimensionamento para demanda de pico e análise de 4 cenários base + aumento gradual de geração até violação de limites (tensão, carregamento, fluxo reverso e perdas).
* **Parte II (Armazenamento - BESS):** Comparação entre armazenamento centralizado e distribuído para mitigação de impactos e absorção de excedentes.
* **Parte III (Microrrede e Ilhamento):** Simulação de operação ilhada em diferentes cenários e aplicação de corte de carga (*load shedding*) para aumentar o tempo de sobrevivência.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import dss  # OpenDSS Module

# Ajuste o caminho para a pasta onde está o arquivo ieee37.dss
os.chdir(r'C:\Users\brun\Desktop\aff\bla')
mydir = os.getcwd()

dss_engine = dss.DSS
DSSText = dss_engine.Text
DSSCircuit = dss_engine.ActiveCircuit
DSSSolution = dss_engine.ActiveCircuit.Solution
ControlQueue = dss_engine.ActiveCircuit.CtrlQueue

## 1. Configuração do Circuito, Cargas e Limites Operativos

In [2]:
def compile_circuit():
    DSSText.Command = f'Compile "{mydir}/ieee37.dss"'

def set_norm_vminmax_pu(pumin=0.93, pumax=1.05):
    DSSText.Command = f'set normvminpu={pumin}'
    DSSText.Command = f'set normvmaxpu={pumax}'
    print(f'Limites de tensão definidos: {pumin} p.u. a {pumax} p.u.')

def get_all_elements(verbose=False):
    Elenames = DSSCircuit.AllElementNames
    Loadnames = DSSCircuit.Loads.AllNames
    Busnames = DSSCircuit.AllBusNames
    Trafonames = DSSCircuit.Transformers.AllNames
    Linenames = DSSCircuit.Lines.AllNames
    Pvnames = DSSCircuit.PVSystems.AllNames
    Storagenames = DSSCircuit.Storages.AllNames

    if verbose:
        print(f'Elementos: {len(Elenames)} | Cargas: {len(Loadnames)} | Barras: {len(Busnames)}')
        print(f'Trafos: {len(Trafonames)} | Linhas: {len(Linenames)} | PVs: {len(Pvnames)} | Storages: {len(Storagenames)}')

    return Elenames, Loadnames, Busnames, Trafonames, Linenames, Pvnames, Storagenames

def preparar_cargas(Loadnames):
    ind_loads = ['s701a', 's701b', 's701c', 's728', 's737a', 's738a']
    resd_profile = [0.45, 0.40, 0.37, 0.35, 0.35, 0.40, 0.55, 0.65, 0.60, 0.50, 0.45, 0.45, 
                    0.50, 0.48, 0.45, 0.45, 0.50, 0.65, 0.85, 0.95, 1.00, 0.95, 0.80, 0.60]
    ind_profile = [0.30, 0.28, 0.27, 0.27, 0.30, 0.35, 0.50, 0.70, 0.88, 0.95, 1.00, 1.00, 
                   0.90, 0.95, 1.00, 0.98, 0.92, 0.80, 0.65, 0.50, 0.42, 0.38, 0.34, 0.32]

    DSSCircuit.LoadShapes.New('resd_loadshape')
    DSSCircuit.LoadShapes.Npts = 24
    DSSCircuit.LoadShapes.HrInterval = 1
    DSSCircuit.LoadShapes.Pmult = resd_profile

    DSSCircuit.LoadShapes.New('ind_loadshape')
    DSSCircuit.LoadShapes.Npts = 24
    DSSCircuit.LoadShapes.HrInterval = 1
    DSSCircuit.LoadShapes.Pmult = ind_profile

    for load in Loadnames:
        DSSCircuit.SetActiveElement(f'load.{load}')
        if load.lower() in ind_loads:
            DSSCircuit.ActiveElement.Properties('daily').Val = 'ind_loadshape'
        else:
            DSSCircuit.ActiveElement.Properties('daily').Val = 'resd_loadshape'
    print('Loadshapes criados e atribuídos às cargas.')

def set_loadmult(load_factor):
    DSSText.Command = f'set loadmult = {load_factor}'
    print(f'Load multiplier definido para {load_factor}')

## 2. Configuração de Sistemas PV, Armazenamento (BESS) e Ilhamento

In [3]:
def pv_system_setup():
    DSSText.Command = 'New XYcurve.PvsT npts=4 xarray=[0 25 75 100] yarray=[1.2 1 .8 .6]'
    DSSText.Command = 'New XYcurve.Effic npts=4 xarray=[.1 .2 .4 1] yarray=[0.86 .9 .93 .97]'
    DSSText.Command = 'New loadshape.Irrad npts=24 interval=1 mult=[0 0 0 0 0 0 .1 .2 .3 .5 .8 .9 1.0 1.0 .99 .9 .7 .4 .1 0 0 0 0 0]'
    DSSText.Command = 'New Tshape.Temp npts=24 interval=1 temp=[25 25 25 25 25 25 25 25 35 40 45 50 60 60 55 40 35 30 25 25 25 25 25 25]'
    print('Curvas do sistema PV criadas.')

def add_pv_systems(pmpp_x, kva_inv):
    barras_pv = ['728', '733', '737', '740', '718', '720', '722', '742']
    for idx, bus in enumerate(barras_pv, start=1):
        DSSText.Command = (
            f'New PVSystem.PV_{idx} phases=3 bus1={bus}.1.2.3 kv=4.8 conn=delta '
            f'irrad=1.0 pmpp={pmpp_x} kva={kva_inv} temperature=25 pf=1 '
            f'%cutin=0.05 %cutout=0.05 effcurve=Effic P-tcurve=PvsT Daily=Irrad Tdaily=Temp'
        )
    print(f'{len(barras_pv)} sistemas PV criados com pmpp={pmpp_x} kW e kva={kva_inv} kVA.')

def storage_setup():
    DSSText.Command = 'New XYcurve.Eff npts=4 xarray=[.1 .2 .4 1] yarray=[0.86 .9 .93 .97]'
    DSSText.Command = (
        'New LoadShape.dispatch_shape npts=24 interval=1 '
        'mult=[0 0 0 0 0 0 0 0 0 0 -0.3 -0.4 -0.45 -0.45 -0.45 -0.4 -0.15 0.2 0.4 0.6 0.6 0.4 0.2 0]'
    )
    print('Curvas de armazenamento criadas.')

def add_storage(nome, bus, kw, kwh, kv=4.8, phases=3, conn="delta", stored=20, reserve=20, dispmode="follow"):
    cmd = (
        f"New Storage.{nome} phases={phases} bus1={bus} kv={kv} conn={conn} "
        f"kWrated={kw} kWhrated={kwh} %stored={stored} %reserve={reserve} "
        f"effcurve=Eff dispmode={dispmode} pf=1"
    )
    if dispmode == "follow":
        cmd += " daily=dispatch_shape"
    DSSText.Command = cmd

def configurar_bess(tipo="centralizado", stored=20, dispmode="follow"):
    storage_setup()
    if tipo == "centralizado":
        add_storage("StoCentral", "709.1.2.3", 2464, 10000, stored=stored, dispmode=dispmode)
        print("Storage centralizado adicionado na barra 709.")
    elif tipo == "distribuido":
        bess_list = [
            ("Sto1", "709.1.2.3", 1000, 4000),
            ("Sto2", "734.1.2.3", 1464, 6000),
        ]
        for nome, bus, kw, kwh in bess_list:
            add_storage(nome, bus, kw, kwh, stored=stored, dispmode=dispmode)
        print("Storages distribuídos adicionados nas barras 709 e 734.")

def configurar_ilhamento(linhailha="L27", n_corte=0):
    DSSText.Command = f'Open Line.{linhailha} Term=1'
    print(f'Linha {linhailha} aberta (Term=1) para operação ilhada.')

    # Prioridade de corte (menor prioridade primeiro)
    cargas_ilha = [
        ('S736b', 42), ('S741c', 42), ('S732c', 42), ('S734c', 42), ('S735c', 85),
        ('S733a', 85), ('S740c', 85), ('S738a', 126), ('S737a', 140)
    ]
    for nome, kw in cargas_ilha[:n_corte]:
        DSSText.Command = f'Edit Load.{nome} enabled=no'
    if n_corte > 0:
        print(f'Corte de carga aplicado: {n_corte} cargas desligadas.')

## 3. Motor de Simulação de Fluxo de Potência Diário

In [4]:
def run_power_flow(simulation_time, Loadnames, Linenames):
    DSSText.Command = 'reset'
    DSSText.Command = 'set controlmode=static'
    DSSText.Command = 'set mode=daily'
    DSSText.Command = 'set stepsize=1h'
    DSSText.Command = 'set number=1'

    Pvnames = DSSCircuit.PVSystems.AllNames
    Storagenames = DSSCircuit.Storages.AllNames
    onlytrafo = [n for n in DSSCircuit.Transformers.AllNames if 'reg' not in n.lower()]
    T = simulation_time

    Total_Act_Power = np.zeros(T)
    Total_React_Power = np.zeros(T)
    Total_Act_Losses = np.zeros(T)
    Total_React_Losses = np.zeros(T)

    busVpuA = [None] * T
    busVpuB = [None] * T
    busVpuC = [None] * T

    Trafo_Max_Amp = np.zeros((len(onlytrafo), T))
    Trafo_Loading_pu = np.zeros((len(onlytrafo), T))
    Load_kW = np.zeros((len(Loadnames), T))
    Load_kvar = np.zeros((len(Loadnames), T))
    Line_Max_Amp = np.zeros((len(Linenames), T))
    Line_Loading_pu = np.zeros((len(Linenames), T))
    PV_kW = np.zeros((len(Pvnames), T))
    Storage_kW = np.zeros((len(Storagenames), T))
    Storage_kWh_stored = np.zeros((len(Storagenames), T))

    def max_current(elem):
        ncond = elem.NumConductors
        return max(elem.CurrentsMagAng[0:2*ncond:2])

    for h in range(T):
        DSSSolution.Solve()
        if not DSSSolution.Converged:
            print(f"[AVISO] Fluxo não convergiu na hora {h}!")

        Total_Act_Power[h], Total_React_Power[h] = DSSCircuit.TotalPower[0], DSSCircuit.TotalPower[1]
        Total_Act_Losses[h] = DSSCircuit.Losses[0] / 1000.0
        Total_React_Losses[h] = DSSCircuit.Losses[1] / 1000.0

        busVpuA[h] = DSSCircuit.AllNodeVmagPUByPhase(1)
        busVpuB[h] = DSSCircuit.AllNodeVmagPUByPhase(2)
        busVpuC[h] = DSSCircuit.AllNodeVmagPUByPhase(3)

        for i, t in enumerate(onlytrafo):
            DSSCircuit.SetActiveElement(f'Transformer.{t}')
            elem = DSSCircuit.ActiveElement
            imax = max_current(elem)
            Trafo_Max_Amp[i, h] = imax
            Trafo_Loading_pu[i, h] = imax / elem.NormalAmps if elem.NormalAmps > 0 else 0

        for i, l in enumerate(Loadnames):
            DSSCircuit.SetActiveElement(f'Load.{l}')
            Load_kW[i, h] = DSSCircuit.ActiveElement.TotalPowers[0]
            Load_kvar[i, h] = DSSCircuit.ActiveElement.TotalPowers[1]

        for i, l in enumerate(Linenames):
            DSSCircuit.SetActiveElement(f'Line.{l}')
            elem = DSSCircuit.ActiveElement
            imax = max_current(elem)
            Line_Max_Amp[i, h] = imax
            Line_Loading_pu[i, h] = imax / elem.NormalAmps if elem.NormalAmps > 0 else 0

        for i, p in enumerate(Pvnames):
            DSSCircuit.SetActiveElement(f'PVSystem.{p}')
            # Potência injetada é negativa no OpenDSS; invertemos o sinal para geração positiva
            PV_kW[i, h] = -DSSCircuit.ActiveElement.TotalPowers[0]

        for i, s in enumerate(Storagenames):
            DSSCircuit.SetActiveElement(f'Storage.{s}')
            Storage_kW[i, h] = -DSSCircuit.ActiveElement.TotalPowers[0]
            Storage_kWh_stored[i, h] = float(DSSCircuit.ActiveElement.Properties('kWhstored').Val)

    nodesA = [n for n in DSSCircuit.AllNodeNamesByPhase(1) if not n.startswith('reg')]
    nodesB = [n for n in DSSCircuit.AllNodeNamesByPhase(2) if not n.startswith('reg')]
    nodesC = [n for n in DSSCircuit.AllNodeNamesByPhase(3) if not n.startswith('reg')]

    idxA = [i for i, n in enumerate(DSSCircuit.AllNodeNamesByPhase(1)) if not n.startswith('reg')]
    idxB = [i for i, n in enumerate(DSSCircuit.AllNodeNamesByPhase(2)) if not n.startswith('reg')]
    idxC = [i for i, n in enumerate(DSSCircuit.AllNodeNamesByPhase(3)) if not n.startswith('reg')]

    df_VA = pd.DataFrame(np.vstack(busVpuA)[:, idxA], columns=nodesA)
    df_VB = pd.DataFrame(np.vstack(busVpuB)[:, idxB], columns=nodesB)
    df_VC = pd.DataFrame(np.vstack(busVpuC)[:, idxC], columns=nodesC)

    return {
        "Total_Act_Power": Total_Act_Power,
        "Total_React_Power": Total_React_Power,
        "Total_Act_Losses": Total_Act_Losses,
        "Total_React_Losses": Total_React_Losses,
        "df_VA": df_VA,
        "df_VB": df_VB,
        "df_VC": df_VC,
        "onlytrafo": onlytrafo,
        "Trafo_Max_Amp": Trafo_Max_Amp,
        "Trafo_Loading_pu": Trafo_Loading_pu,
        "Loadnames": Loadnames,
        "Load_kW": Load_kW,
        "Load_kvar": Load_kvar,
        "Linenames": Linenames,
        "Line_Max_Amp": Line_Max_Amp,
        "Line_Loading_pu": Line_Loading_pu,
        "Pvnames": Pvnames,
        "PV_kW": PV_kW,
        "Storagenames": Storagenames,
        "Storage_kW": Storage_kW,
        "Storage_kWh_stored": Storage_kWh_stored,
    }

## 4. Funções de Métricas, Violações e Visualização

In [5]:
def avaliar_metricas(res, v_min_lim=0.93, v_max_lim=1.05):
    print(f"=== RESUMO: {res['nome']} ===")
    # No OpenDSS, potência negativa na fonte indica consumo da rede; positiva indica fluxo reverso
    p_sub = -res["Total_Act_Power"]
    q_sub = -res["Total_React_Power"]

    kwh_total = np.sum(p_sub)
    kvarh_total = np.sum(q_sub)
    kwh_loss = np.sum(res["Total_Act_Losses"])
    fluxo_reverso_max = np.min(p_sub)  # valores < 0 significam injeção na subestação

    print(f"Energia líquida da subestação: {kwh_total:.2f} kWh | {kvarh_total:.2f} kvarh")
    print(f"Perdas ativas totais: {kwh_loss:.2f} kWh ({abs(kwh_loss/kwh_total)*100:.2f}%)" if kwh_total != 0 else f"Perdas ativas totais: {kwh_loss:.2f} kWh")
    if fluxo_reverso_max < 0:
        print(f"[FLUXO REVERSO] Máximo de {abs(fluxo_reverso_max):.2f} kW injetados na subestação!")
    else:
        print("Sem fluxo reverso na subestação.")

    # Tensões
    for fase, df in [("A", res["df_VA"]), ("B", res["df_VB"]), ("C", res["df_VC"])]:
        df_valid = df.replace(0, np.nan)
        vmin = df_valid.min().min()
        vmax = df_valid.max().max()
        print(f"Tensão Fase {fase}: Mín = {vmin:.4f} p.u. | Máx = {vmax:.4f} p.u.")
        if vmin < v_min_lim or vmax > v_max_lim:
            print(f"  -> [VIOLAÇÃO DE TENSÃO] Fase {fase} fora da faixa [{v_min_lim}, {v_max_lim}] p.u.")

    # Carregamento
    max_line_pu = np.max(res["Line_Loading_pu"]) if res["Line_Loading_pu"].size > 0 else 0
    max_trafo_pu = np.max(res["Trafo_Loading_pu"]) if res["Trafo_Loading_pu"].size > 0 else 0
    print(f"Carregamento máximo em Linhas: {max_line_pu*100:.2f}%")
    if max_line_pu > 1.0:
        print("  -> [SOBRECARGA] Há linhas acima de 100% da corrente nominal!")
    print(f"Carregamento máximo em Transformadores: {max_trafo_pu*100:.2f}%")
    if max_trafo_pu > 1.0:
        print("  -> [SOBRECARGA] Há transformadores acima de 100% da corrente nominal!")
    print("-" * 50)

def plot_envoltoria_tensao(res):
    fases = {'Fase A': res['df_VA'], 'Fase B': res['df_VB'], 'Fase C': res['df_VC']}
    cores = ['tab:blue', 'tab:orange', 'tab:green']
    fig, ax = plt.subplots(figsize=(8, 4))

    for (nome, df), cor in zip(fases.items(), cores):
        df_clean = df.replace(0, np.nan)
        vmax = df_clean.max(axis=1)
        vmin = df_clean.min(axis=1)
        ax.fill_between(df_clean.index, vmin, vmax, color=cor, alpha=0.25, label=nome)
        ax.plot(df_clean.index, vmax, color=cor, linewidth=0.9)
        ax.plot(df_clean.index, vmin, color=cor, linewidth=0.9)

    ax.axhline(1.05, color='r', linestyle='--', linewidth=1, label='Limites (0.93 / 1.05 p.u.)')
    ax.axhline(0.93, color='r', linestyle='--', linewidth=1)
    ax.set_title(f"Envoltória de Tensão por Fase — {res['nome']}", fontsize=10)
    ax.set_xlabel('Hora', fontsize=9)
    ax.set_ylabel('Tensão (p.u.)', fontsize=9)
    ax.set_xticks(range(0, 25, 3))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout()
    plt.show()

def plot_subestacao(res):
    p_sub = -res["Total_Act_Power"]
    q_sub = -res["Total_React_Power"]
    losses = res["Total_Act_Losses"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Desempenho na Subestação e Perdas — {res['nome']}", fontsize=11)

    axes[0].plot(p_sub, color='tab:blue', linewidth=1.5)
    axes[0].axhline(0, color='k', linestyle=':', linewidth=0.8)
    axes[0].set_title('Potência Ativa na Subestação (kW)', fontsize=9)
    axes[0].set_xlabel('Hora')
    axes[0].set_ylabel('kW')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(q_sub, color='tab:orange', linewidth=1.5)
    axes[1].set_title('Potência Reativa na Subestação (kvar)', fontsize=9)
    axes[1].set_xlabel('Hora')
    axes[1].set_ylabel('kvar')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(losses, color='tab:red', linewidth=1.5)
    axes[2].set_title('Perdas Ativas Totais (kW)', fontsize=9)
    axes[2].set_xlabel('Hora')
    axes[2].set_ylabel('kW')
    axes[2].grid(True, alpha=0.3)

    for ax in axes:
        ax.set_xticks(range(0, 25, 3))
    plt.tight_layout()
    plt.show()

def plot_carregamento(res):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f"Carregamento de Linhas e Transformadores — {res['nome']}", fontsize=11)

    df_lines = pd.DataFrame(res["Line_Loading_pu"].T * 100, columns=res["Linenames"])
    df_lines.plot(ax=axes[0], legend=False, linewidth=1)
    axes[0].axhline(100, color='r', linestyle='--', linewidth=1)
    axes[0].set_title('Carregamento Máximo nas Linhas (%)', fontsize=9)
    axes[0].set_xlabel('Hora')
    axes[0].set_ylabel('% Nominal')
    axes[0].grid(True, alpha=0.3)

    df_trafos = pd.DataFrame(res["Trafo_Loading_pu"].T * 100, columns=res["onlytrafo"])
    df_trafos.plot(ax=axes[1], linewidth=1.2)
    axes[1].axhline(100, color='r', linestyle='--', linewidth=1)
    axes[1].set_title('Carregamento Máximo nos Transformadores (%)', fontsize=9)
    axes[1].set_xlabel('Hora')
    axes[1].set_ylabel('% Nominal')
    axes[1].legend(fontsize=7)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_pv_e_storage(res):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f"Geração PV e Operação do BESS — {res['nome']}", fontsize=11)

    if res["PV_kW"].size > 0:
        df_pv = pd.DataFrame(res["PV_kW"].T, columns=res["Pvnames"])
        df_pv.plot(ax=axes[0], linewidth=1.2)
        axes[0].set_title('Potência Ativa PV (kW)', fontsize=9)
        axes[0].set_xlabel('Hora')
        axes[0].set_ylabel('kW')
        axes[0].legend(fontsize=7)
        axes[0].grid(True, alpha=0.3)
    else:
        axes[0].text(0.5, 0.5, 'Sem sistemas PV ativos', ha='center', va='center')

    if res["Storage_kW"].size > 0:
        df_sto = pd.DataFrame(res["Storage_kW"].T, columns=res["Storagenames"])
        df_sto.plot(ax=axes[1], linewidth=1.5)
        axes[1].axhline(0, color='k', linestyle=':', linewidth=0.8)
        axes[1].set_title('Potência Storage (+ Descarga / - Carga) [kW]', fontsize=9)
        axes[1].set_xlabel('Hora')
        axes[1].set_ylabel('kW')
        axes[1].legend(fontsize=8)
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, 'Sem Storage ativo', ha='center', va='center')

    plt.tight_layout()
    plt.show()

## 5. Orquestrador Unificado de Cenários

In [6]:
def cenario(
    nome,
    multiplicador_carga=1.0,
    pv_ativo=False,
    pmpp=3000,
    kva_inv=3000,
    storage=False,
    storage_tipo="centralizado",
    storage_carga=20,
    ilhamento=False,
    linhailha="L27",
    corte_carga=0,
    tempo=24
):
    compile_circuit()
    set_norm_vminmax_pu(0.93, 1.05)

    Elenames, Loadnames, Busnames, Trafonames, Linenames, _, _ = get_all_elements()
    preparar_cargas(Loadnames)
    set_loadmult(multiplicador_carga)

    if pv_ativo:
        pv_system_setup()
        add_pv_systems(pmpp, kva_inv)

    if storage:
        dispmode = "default" if ilhamento else "follow"
        configurar_bess(tipo=storage_tipo, stored=storage_carga, dispmode=dispmode)

    if ilhamento:
        n_corte = corte_carga if isinstance(corte_carga, int) else (3 if corte_carga else 0)
        configurar_ilhamento(linhailha=linhailha, n_corte=n_corte)

    res = run_power_flow(simulation_time=tempo, Loadnames=Loadnames, Linenames=Linenames)
    res["nome"] = nome

    avaliar_metricas(res)
    return res

## Parte I — Impactos da Geração Fotovoltaica Distribuída (Cenários 1 a 5)

In [10]:
# Cenário 1: Sem PV, carga de pico (loadmult = 1.0)
resultado_1 = cenario(nome="Cenário 1 (Base Pico)", multiplicador_carga=1.0, pv_ativo=False)

# Cenário 2: Sem PV, carga leve (loadmult = 0.2)
resultado_2 = cenario(nome="Cenário 2 (Base Leve 20%)", multiplicador_carga=0.2, pv_ativo=False)

# Cenário 3: Com PV, carga leve (loadmult = 0.2)
resultado_3 = cenario(nome="Cenário 3 (PV + Carga 20%)", multiplicador_carga=0.2, pv_ativo=True, pmpp=300, kva_inv=300)

# Cenário 4: Com PV dimensionado para demanda de pico, carga cheia (loadmult = 1.0)
resultado_4 = cenario(nome="Cenário 4 (PV + Carga 100%)", multiplicador_carga=1.0, pv_ativo=True, pmpp=300, kva_inv=300)

# Cenário 5: Aumento gradual de PV para identificar limite de violação
#resultado_5 = cenario(nome="Cenário 5 (PV Elevado - Teste Violação)", multiplicador_carga=1.0, pv_ativo=True, pmpp=1500, kva_inv=1500)

Limites de tensão definidos: 0.93 p.u. a 1.05 p.u.
Loadshapes criados e atribuídos às cargas.
Load multiplier definido para 1.0


DSSException: (#33003) Invalid property name "kWhstored" for "Line.jumper"

In [9]:
# Visualização dos resultados da Parte I
for r in [resultado_1, resultado_4, resultado_5]:
    plot_envoltoria_tensao(r)
    plot_subestacao(r)
    plot_carregamento(r)

NameError: name 'resultado_1' is not defined

## Parte II — Armazenamento de Energia: Centralizado vs. Distribuído (Cenários 6 e 7)

In [ ]:
# Cenário 6: PV elevado + Armazenamento Distribuído
resultado_6 = cenario(
    nome="Cenário 6 (PV + BESS Distribuído)",
    multiplicador_carga=1.0,
    pv_ativo=True,
    pmpp=1500,
    kva_inv=1500,
    storage=True,
    storage_tipo="distribuido"
)

# Cenário 7: PV elevado + Armazenamento Centralizado
resultado_7 = cenario(
    nome="Cenário 7 (PV + BESS Centralizado)",
    multiplicador_carga=1.0,
    pv_ativo=True,
    pmpp=1500,
    kva_inv=1500,
    storage=True,
    storage_tipo="centralizado"
)

for r in [resultado_6, resultado_7]:
    plot_envoltoria_tensao(r)
    plot_subestacao(r)
    plot_pv_e_storage(r)

## Parte III — Operação Ilhada (Microrrede) e Corte de Carga (Cenários 8 e 9)

In [ ]:
# Cenário 8: Ilhamento sem PV, com BESS e corte de carga
resultado_8 = cenario(
    nome="Cenário 8 (Ilha sem PV + BESS + Corte)",
    multiplicador_carga=1.0,
    ilhamento=True,
    pv_ativo=False,
    storage=True,
    storage_tipo="distribuido",
    storage_carga=100,
    corte_carga=4
)

# Cenário 9: Ilhamento com PV, BESS carregado a 100% e corte de carga
resultado_9 = cenario(
    nome="Cenário 9 (Ilha com PV + BESS + Corte)",
    multiplicador_carga=1.0,
    ilhamento=True,
    pv_ativo=True,
    pmpp=1900,
    kva_inv=1900,
    storage=True,
    storage_tipo="distribuido",
    storage_carga=100,
    corte_carga=3
)

for r in [resultado_8, resultado_9]:
    plot_envoltoria_tensao(r)
    plot_pv_e_storage(r)